# Aprendizado de Máquina — Aula prática 04

## Métodos Não Paramétricos (KNN)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Nas Aulas 01 e 02 nós **escolhíamos a forma** de $r$ — uma reta, um polinômio de
grau 5 — e estimávamos alguns coeficientes. A pergunta desta aula é a saída
natural:

> **e se eu não quiser supor forma alguma para $r(x)$?**

A resposta é deixar os dados ditarem a forma, olhando só para a **vizinhança** do
ponto onde se quer prever. Vamos construir dois caminhos para isso — uma base de
*splines* e o KNN — e, o mais importante, medir o que eles cobram por essa
liberdade. O último experimento do notebook mostra que a conta pode ficar
salgada, e é a deixa para a Aula 05.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- montar uma base de *splines* à mão e reconhecer o que a torna **local**;
- implementar KNN e ver o $k$ funcionar como botão de flexibilidade — só que ao
  contrário do grau do polinômio;
- reproduzir o experimento do [ISLP] §3.5 em que a regressão linear, apesar de
  errada, ganha do KNN quando a dimensão cresce.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos são o estimador KNN do `scikit-learn` e o `SplineTransformer`,
que monta uma base de *splines* pronta.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. A mesma população de sempre

$$X \sim \mathrm{Unif}[-3,3], \qquad Y = \sin(1{,}5X) + 0{,}3X + \varepsilon,
  \qquad \varepsilon \sim N(0, 0{,}7^2).$$

Continuamos na população da Aula 01 pelo mesmo motivo de sempre: aqui conhecemos
$r$, então dá para separar viés de variância — que é exatamente o que esta aula
precisa medir.

In [ ]:
def r(x):
    return np.sin(1.5 * x) + 0.3 * x


SIGMA = 0.7
A, B = -3.0, 3.0
N_TR = 50


def amostra(n, rng):
    x = rng.uniform(A, B, size=n)
    return x, r(x) + rng.normal(0, SIGMA, size=n)


rng = np.random.default_rng(0)
x, y = amostra(N_TR, rng)
X = x.reshape(-1, 1)
grade = np.linspace(A, B, 500)

---
## 3. Bases: ganhando flexibilidade sem sair do linear

A primeira estratégia não precisa de método novo: basta **inventar atributos**.
Se ajustarmos

$$g(x) = \sum_{j=1}^{I} \beta_j\, \phi_j(x),$$

o modelo continua linear nos $\beta_j$ e sai por mínimos quadrados. Foi o que
fizemos na Aula 01 com $\phi_j(x)=x^j$.

Os *splines* trocam os monômios por uma base melhor. Um *spline* de grau $k$ com
nós $t_1 < \dots < t_p$ é um polinômio por partes de grau $k$, colado nos nós com
derivadas até ordem $k-1$ contínuas. A base mais fácil de entender é a **base
truncada**:

$$1,\ x,\ \dots,\ x^k,\qquad (x-t_j)_+^k,\ j=1,\dots,p,
\qquad \text{onde } (u)_+ = \max(u, 0).$$

O truque está na segunda metade. A função $(x-t_j)_+^k$ é **identicamente nula à
esquerda de $t_j$**: acrescentá-la não muda nada antes do nó, e permite mudar a
curvatura depois dele. E como ela e suas $k-1$ primeiras derivadas se anulam em
$t_j$, a emenda sai suave de graça. Cada nó compra exatamente um grau de
liberdade, colocado onde você quiser.

In [ ]:
def base_truncada(x, nos, grau=3):
    colunas = [x ** j for j in range(1, grau + 1)]
    colunas += [np.clip(x - t, 0, None) ** grau for t in nos]
    return np.column_stack(colunas)


nos = np.array([-1.5, 0.0, 1.5])
spline = skl.LinearRegression().fit(base_truncada(x, nos), y)

# um polinomio global com o MESMO numero de parametros, para comparar
I = base_truncada(x, nos).shape[1]
poli = skl.LinearRegression().fit(np.column_stack([x ** j for j in range(1, I + 1)]), y)

fig, ax = subplots(figsize=(5.6, 3.3))
ax.scatter(x, y, s=16, color="gray", alpha=0.8)
ax.plot(grade, r(grade), color="green", lw=1.8, label="r(x)")
ax.plot(grade, spline.predict(base_truncada(grade, nos)), color="crimson", lw=1.5,
        label=f"spline cubico, {len(nos)} nos")
ax.plot(grade, poli.predict(np.column_stack([grade ** j for j in range(1, I + 1)])),
        color="steelblue", lw=1.5, ls="--", label=f"polinomio global grau {I}")
for t in nos:
    ax.axvline(t, color="gray", lw=0.7, ls=":")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_ylim(-3, 3)
ax.legend(fontsize=7.5)
print(f"os dois modelos tem {I} parametros (fora o intercepto)")

Mesmo número de parâmetros, ajustes diferentes. A diferença que interessa não é
estética: é que o *spline* é **local** e o polinômio global não é.

Vamos ver isso acontecer. Empurramos o $y$ do ponto mais à direita bem para cima —
doze unidades, um exagero deliberado — e olhamos o que muda **do outro lado do
domínio**. Como os dois ajustes são lineares em $y$, o tamanho do empurrão não
altera a *proporção* entre eles: só torna o efeito grande o bastante para enxergar.

E fazemos isso duas vezes, com 3 nós e com 10, porque a comparação entre as duas
é que responde a pergunta.

In [ ]:
DELTA = 12.0                       # exagero deliberado, so para o efeito aparecer
i_direita = np.argmax(x)
y_mexido = y.copy()
y_mexido[i_direita] += DELTA

nos_muitos = np.linspace(A, B, 12)[1:-1]        # 10 nos igualmente espacados
esquerda = grade < -1.0

fig, axes = subplots(2, 2, figsize=(9.6, 5.8), sharex=True)

for col, (ns, rotulo) in enumerate([(nos, "3 nos"), (nos_muitos, "10 nos")]):
    Ic = base_truncada(x, ns).shape[1]
    mono = lambda v: np.column_stack([v ** j for j in range(1, Ic + 1)])

    sp0 = skl.LinearRegression().fit(base_truncada(x, ns), y)
    sp1 = skl.LinearRegression().fit(base_truncada(x, ns), y_mexido)
    po0 = skl.LinearRegression().fit(mono(x), y)
    po1 = skl.LinearRegression().fit(mono(x), y_mexido)

    d_sp = (sp1.predict(base_truncada(grade, ns))
            - sp0.predict(base_truncada(grade, ns)))
    d_po = po1.predict(mono(grade)) - po0.predict(mono(grade))

    a = axes[0, col]
    a.scatter(x, y, s=11, color="gray", alpha=0.6)
    a.scatter([x[i_direita]], [y_mexido[i_direita]], s=70, color="black",
              marker="*", zorder=6)
    a.plot(grade, sp0.predict(base_truncada(grade, ns)), color="crimson",
           lw=1.1, ls="--", alpha=0.6)
    a.plot(grade, sp1.predict(base_truncada(grade, ns)), color="crimson",
           lw=1.7, label=f"spline, {rotulo}")
    a.plot(grade, po0.predict(mono(grade)), color="steelblue", lw=1.1, ls="--", alpha=0.6)
    a.plot(grade, po1.predict(mono(grade)), color="steelblue", lw=1.7,
           label=f"polinomio grau {Ic}")
    for t in ns:
        a.axvline(t, color="gray", lw=0.5, ls=":")
    a.set_ylim(-4, 4)
    a.set_title(f"{rotulo}: antes (tracejado) e depois", fontsize=9)
    a.legend(fontsize=7)

    b = axes[1, col]
    b.axhline(0, color="black", lw=0.6)
    b.plot(grade, d_sp, color="crimson", lw=1.7, label="spline")
    b.plot(grade, d_po, color="steelblue", lw=1.7, label="polinomio")
    for t in ns:
        b.axvline(t, color="gray", lw=0.5, ls=":")
    b.set_ylim(-1.2, 1.2)
    b.set_xlabel("x")
    b.legend(fontsize=7)

    m_sp, m_po = np.abs(d_sp)[esquerda].max(), np.abs(d_po)[esquerda].max()
    b.set_title(f"a mudanca, por x — o polinomio muda {m_po / m_sp:.0f}x mais "
                f"em x < -1", fontsize=9)
    print(f"{rotulo:>7} ({Ic:2d} parametros): maior mudanca em x < -1   "
          f"spline {m_sp:.4f}   polinomio {m_po:.4f}   razao {m_po / m_sp:.1f}x")

axes[0, 0].set_ylabel("y")
axes[1, 0].set_ylabel("mudanca no ajuste")
fig.tight_layout()

A linha de baixo é onde está a resposta.

Com **3 nós** as duas curvas de mudança são quase a mesma curva: o ponto lá da
ponta direita mexeu no ajuste em $x=-3$ praticamente igual nos dois casos, e o
*spline* só ganha por um fator de 3. Com **10 nós** o quadro é outro — a curva
vermelha fica colada no zero até o último nó e só então acorda, enquanto a azul
oscila pelo domínio inteiro. O fator vai a 25.

A moral, então, não é "*spline* é local". É que **a localidade se compra com nós**.
Um *spline* cúbico com 3 nós ainda contém todas as cúbicas, e cúbica é função
global: levantar um ponto na borda inclina a cúbica inteira, e o efeito atravessa o
domínio. Cada nó a mais acrescenta uma direção que só existe depois dele, e é isso
que vai isolando um lado do outro.

Cuidado com uma explicação tentadora e errada: o problema **não** é a base truncada
carregar os monômios. O ajuste depende só do *espaço gerado*, não da base escolhida
para descrevê-lo. É justamente por isso que dá para trocar de base sem trocar de
modelo — e é o que as bibliotecas fazem, usando **B-splines**, que geram o mesmo
espaço com contas numericamente estáveis. *Mesmo espaço* quer dizer ajuste
idêntico, e isso é conferível.

In [ ]:
bs = SplineTransformer(degree=3, knots=np.r_[A, nos, B].reshape(-1, 1),
                       extrapolation="continue", include_bias=False)
modelo_bs = skl.LinearRegression().fit(bs.fit_transform(X), y)

f_trunc = spline.predict(base_truncada(grade, nos))
f_bs = modelo_bs.predict(bs.transform(grade.reshape(-1, 1)))
print(f"colunas: base truncada = {base_truncada(x, nos).shape[1]}, "
      f"B-spline = {bs.transform(X).shape[1]}")
print(f"maior diferenca entre os dois ajustes: {np.abs(f_trunc - f_bs).max():.3e}")

O número de nós é mais um botão de complexidade, e gira como todos os outros.
Comparamos três ajustes na mesma amostra — 1 nó, os 3 de cima e 10 igualmente
espaçados — medindo erro de treino e erro numa amostra nova de 20 mil pontos.

In [ ]:
x_novo, y_novo = amostra(20_000, np.random.default_rng(99))

configs = [(np.array([0.0]), "1 no"),
           (nos, "3 nos"),
           (np.linspace(A, B, 12)[1:-1], "10 nos")]

fig, ax = subplots(figsize=(6.0, 3.4))
ax.scatter(x, y, s=16, color="gray", alpha=0.8)
ax.plot(grade, r(grade), color="green", lw=1.8, label="r(x)")

for (ns, rotulo), cor in zip(configs, ("steelblue", "crimson", "darkorange")):
    m = skl.LinearRegression().fit(base_truncada(x, ns), y)
    tr = np.mean((y - m.predict(base_truncada(x, ns))) ** 2)
    te = np.mean((y_novo - m.predict(base_truncada(x_novo, ns))) ** 2)
    print(f"{rotulo:>7} ({base_truncada(x, ns).shape[1]:2d} colunas): "
          f"treino {tr:.4f}   teste {te:.4f}")
    ax.plot(grade, m.predict(base_truncada(grade, ns)), color=cor, lw=1.4, label=rotulo)

ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(fontsize=8)

O padrão da Aula 01, de novo, agora com o número de nós no lugar do grau:

| nós | treino | teste |
| --- | --- | --- |
| 1 | $0{,}5656$ | $0{,}5850$ |
| 3 | $0{,}4947$ | $0{,}5052$ |
| 10 | $0{,}3924$ | $0{,}6297$ |

Com **1 nó** o erro de treino já é o maior dos três: não é falta de sorte, é falta
de flexibilidade — quatro colunas não dão conta de duas curvas e meia de seno. É o
subajuste, e ele aparece nos dois números ao mesmo tempo.

Com **10 nós** o erro de treino cai a $0{,}39$, o menor dos três, e o de teste sobe
para $0{,}63$, o maior dos três. Treze colunas para 50 pontos: o ajuste começa a
seguir o ruído. Repare que $0{,}39$ está **abaixo** de $\sigma^2=0{,}49$, que é o
piso do risco de verdade — o sinal de otimismo que a Aula 03 nomeou.

O meio-termo de 3 nós é o único cujos dois números praticamente coincidem, e é o
que a validação cruzada escolheria sem ver o teste.

---
## 4. $k$ vizinhos mais próximos

O estimador mais direto que existe: para prever em $x$, tire a média das
respostas dos $k$ pontos de treino mais próximos.

$$\widehat r(x) = \frac{1}{k}\sum_{i \in \mathcal{N}_x} Y_i .$$

Repare no que essa fórmula **não** tem: nenhum parâmetro a estimar, nenhuma
otimização. O "treino" do KNN consiste em guardar os dados.

In [ ]:
fig, axes = subplots(1, 3, figsize=(11.0, 4.2), sharey=True)
for ax, k, rotulo in zip(axes, [1, 9, 40], ["superajuste", "equilibrio", "subajuste"]):
    m = KNeighborsRegressor(n_neighbors=k).fit(X, y)
    ax.scatter(x, y, s=20, color="gray", alpha=0.7)
    ax.plot(grade, r(grade), color="green", lw=2.0, label="r(x)")
    ax.plot(grade, m.predict(grade.reshape(-1, 1)), color="crimson", lw=1.9,
            label="KNN")
    ax.set_title(f"k = {k} — {rotulo}", fontsize=12)
    ax.set_xlabel("x", fontsize=11); ax.set_ylim(-3.2, 3.2)
    ax.tick_params(labelsize=10)
axes[0].set_ylabel("y", fontsize=11)
axes[0].legend(fontsize=10, loc="upper left")
fig.tight_layout()

Compare com a Aula 01: é o mesmo trio subajuste–equilíbrio–superajuste, com o
botão girando **ao contrário**. Lá o grau *grande* era o flexível; aqui é o $k$
*pequeno*. Vale fixar isso, porque é fonte permanente de confusão.

E note o formato da curva com $k=1$: uma escada. O KNN é constante por partes por
construção — ele nunca vai produzir uma curva suave, por mais dados que receba.

### Preguiçoso de verdade?

"O treino do KNN consiste em guardar os dados" é verdade sobre o **método**: não há
parâmetro a estimar, e toda a conta acontece na hora de prever. Mas será que é
verdade sobre a **biblioteca**? Dá para cronometrar — e a resposta tem uma surpresa.

Vamos medir três configurações num conjunto de 20 mil pontos, prevendo em outros 20
mil. Duas são o mesmo KNN, mudando só como ele procura os vizinhos.

In [ ]:
import time
from sklearn.neighbors import KDTree

rng_g = np.random.default_rng(1)
x_g, y_g = amostra(20_000, rng_g)
X_g = x_g.reshape(-1, 1)
X_novo = np.linspace(A, B, 20_000).reshape(-1, 1)


def crono(f, rep=7):
    """Mediana de `rep` execucoes, em ms -- uma medida so seria ruido."""
    ts = []
    for _ in range(rep):
        t0 = time.perf_counter()
        f()
        ts.append(time.perf_counter() - t0)
    return 1000 * np.median(ts)


modelos = [
    ("KNN, algorithm='brute'", KNeighborsRegressor(n_neighbors=9, algorithm="brute")),
    ("KNN, algorithm='kd_tree'", KNeighborsRegressor(n_neighbors=9, algorithm="kd_tree")),
    ("regressao linear", skl.LinearRegression()),
]

print(f"{'':26} {'ajuste':>10} {'predicao':>11}")
for nome, modelo in modelos:
    t_fit = crono(lambda: modelo.fit(X_g, y_g))
    modelo.fit(X_g, y_g)
    t_pred = crono(lambda: modelo.predict(X_novo))
    print(f"{nome:26} {t_fit:>7.2f} ms {t_pred:>8.2f} ms")

# a arvore muda o tempo, nao o resultado
p_bru = KNeighborsRegressor(n_neighbors=9, algorithm="brute").fit(X_g, y_g).predict(X_novo)
p_kdt = KNeighborsRegressor(n_neighbors=9, algorithm="kd_tree").fit(X_g, y_g).predict(X_novo)
print(f"\npredicoes de 'brute' e 'kd_tree' sao identicas? {np.array_equal(p_bru, p_kdt)}")

Os tempos absolutos dependem da máquina e variam de uma execução para outra — os
seus não vão bater com estes. O que é estável, e o que interessa, são as
**proporções** entre as linhas.

**O KNN pode ser preguiçoso, e quem cumpre a promessa é o `brute`.** Ali o ajuste é
literalmente guardar os dados — uma fração de milissegundo, **menos que a regressão
linear**, que ainda precisa resolver um sistema de mínimos quadrados. O preço vem
depois: cada predição varre a amostra inteira, e as 20 mil levam algo como duzentos
milissegundos.

**Mas o padrão não é preguiçoso.** Com `algorithm='auto'` — que aqui resolve para
`kd_tree` — o `fit` **constrói uma árvore de busca**, e é ela que consome quase todo
o tempo: as duas últimas linhas mostram que montar a KDTree responde por ~90% do
ajuste, enquanto copiar os dados não chega a 1%. É por isso que o "ajuste" do KNN
aparece **maior** que o da regressão linear, o que parece contradizer tudo o que
dissemos. Não contradiz: o que está sendo medido aí não é o método, é a biblioteca
fazendo um investimento.

**E é um bom investimento.** Os poucos milissegundos da árvore derrubam a predição
em **cerca de dez vezes**. Trocar uma varredura $O(n)$ por uma descida $O(\log n)$
em cada busca se paga já na primeira leva de predições — e a última linha confirma
que o resultado é o mesmo: as predições de `brute` e `kd_tree` são **idênticas**, bit a bit. A árvore
muda o tempo, nunca a resposta.

O ponto original sobrevive, agora com a ressalva certa: **o custo do KNN mora na
predição, não no ajuste**. Some as duas colunas de qualquer linha do KNN e compare
com a da regressão linear, cuja predição é a mais barata da tabela por duas ordens
de grandeza. Isso importa em produção, onde se treina uma vez e se prevê milhões de
vezes.

> **De passagem.** Você acabou de conhecer o `algorithm`, um daqueles
> hiperparâmetros que ficam escondidos atrás do `auto`. Ele decide se o KNN é
> preguiçoso ou não — e vale saber que `auto` só monta a árvore quando $p \le 15$;
> daí para cima ele desiste e volta para o `brute`. Em dimensão alta a árvore deixa
> de ajudar, o que é a maldição da dimensionalidade aparecendo onde menos se
> espera: no cronômetro. É assunto da Aula 05.

### Escolhendo $k$ com a ferramenta da Aula 03

In [ ]:
ks = np.array([1, 2, 3, 5, 8, 12, 20, 30, 40])
dobras = skm.KFold(5, shuffle=True, random_state=0)

cv = np.array([-skm.cross_val_score(KNeighborsRegressor(n_neighbors=k), X, y,
                                    cv=dobras, scoring="neg_mean_squared_error").mean()
               for k in ks])
k_cv = ks[np.argmin(cv)]

# o risco verdadeiro de cada k, que so a simulacao entrega
def risco_knn(ks, n_rep=200, semente=11):
    rng = np.random.default_rng(semente)
    x0 = np.linspace(A, B, 400)
    r0 = r(x0)
    X0 = x0.reshape(-1, 1)
    soma = np.zeros(len(ks))
    for _ in range(n_rep):
        xb, yb = amostra(N_TR, rng)
        for j, k in enumerate(ks):
            pred = KNeighborsRegressor(n_neighbors=k).fit(xb.reshape(-1, 1), yb).predict(X0)
            soma[j] += np.mean((pred - r0) ** 2)
    return soma / n_rep + SIGMA ** 2


verdade = risco_knn(ks)
print(f"k escolhido pela CV      : {k_cv}")
print(f"k que minimiza o risco   : {ks[np.argmin(verdade)]}")

In [ ]:
fig, ax = subplots(figsize=(5.4, 3.2))
ax.plot(ks, verdade, "o-", ms=4, color="crimson", label="risco verdadeiro")
ax.plot(ks, cv, "s-", ms=4, color="steelblue", label="validacao cruzada")
ax.axhline(SIGMA ** 2, ls="--", lw=1, color="green", label="sigma^2")
ax.set_xlabel("k (numero de vizinhos)"); ax.set_ylabel("erro quadratico medio")
ax.set_xticks(ks); ax.legend(fontsize=8)

Pense antes de rodar: com dez vezes mais dados, os 9 vizinhos mais próximos ficam
**mais** próximos — menos viés — e a média de mais respostas fica mais estável —
menos variância. Os dois efeitos parecem puxar na mesma direção. Vamos ver para
onde o $k$ ótimo anda de fato.

In [ ]:
def risco_knn_n(ks_, n_tr, n_rep=100, semente=11):
    """Risco verdadeiro do KNN, para um tamanho de treino qualquer."""
    rng_ = np.random.default_rng(semente)
    x0 = np.linspace(A, B, 400)
    r0, X0 = r(x0), np.linspace(A, B, 400).reshape(-1, 1)
    soma = np.zeros(len(ks_))
    for _ in range(n_rep):
        xb, yb = amostra(n_tr, rng_)
        for j, k in enumerate(ks_):
            pred = KNeighborsRegressor(n_neighbors=k).fit(xb.reshape(-1, 1), yb).predict(X0)
            soma[j] += np.mean((pred - r0) ** 2)
    return soma / n_rep + SIGMA ** 2


fig, ax = subplots(figsize=(5.6, 3.3))
for n_tr, ks_n, cor in [(50, np.array([1, 2, 3, 5, 8, 12, 20, 30, 40]), "crimson"),
                        (500, np.array([1, 3, 5, 9, 15, 25, 40, 60, 90, 130]), "steelblue")]:
    v = risco_knn_n(ks_n, n_tr)
    k_estrela = ks_n[np.argmin(v)]
    print(f"n = {n_tr:3d}:  k* = {k_estrela:3d}   risco minimo {v.min():.4f}"
          f"   k*/n = {k_estrela / n_tr:.3f}")
    ax.plot(ks_n, v, "o-", ms=4, color=cor, label=f"n = {n_tr}  (k* = {k_estrela})")

ax.axhline(SIGMA ** 2, ls="--", lw=1, color="green", label="sigma^2")
ax.set_xscale("log")
ax.set_xlabel("k (escala log)"); ax.set_ylabel("risco verdadeiro")
ax.legend(fontsize=8)

**O $k$ ótimo sobe: de 8 para 60.** Com dez vezes mais dados, vale a pena fazer a
média de sete vezes mais vizinhos.

Mas a razão $k^*/n$ **desce**, de $0{,}16$ para $0{,}12$ — e é aí que os dois
efeitos se separam. O que define o viés não é quantos vizinhos você usa, é quão
longe está o mais distante deles. Com $n=500$, os 60 vizinhos mais próximos ocupam
um pedaço menor do eixo $x$ do que os 8 ocupavam com $n=50$. O estimador fica
**mais local em $x$ e mais estável em $y$ ao mesmo tempo**, e é isso que derruba o
risco de $0{,}59$ para $0{,}50$ — encostando no piso $\sigma^2=0{,}49$.

Vale notar a forma da curva azul: entre $k=25$ e $k=90$ o risco anda entre
$0{,}503$ e $0{,}516$. Com amostra grande a escolha de $k$ para de ser crítica, o
que é a mesma observação que a Aula 03 fez sobre a curva plana de $\lambda$.

---
## 5. Quando o paramétrico, mesmo errado, ganha

Até aqui o não paramétrico só teve vantagens: não supõe forma, acompanha o que os
dados mostrarem. Falta a conta.

O experimento é o do [ISLP] §3.5 (a Figura 3.20 deles). A verdade é **não
linear** — é o nosso $r(x_1)$ — e depende de **uma única** covariável. As outras
$p-1$ são ruído puro, irrelevantes. A regressão linear está errada por
construção: ela nem consegue representar a senoide. Mesmo assim, vejamos quem
ganha à medida que $p$ cresce.

In [ ]:
def experimento_dimensao(p, ks, n_tr=300, n_te=4000, n_rep=30, semente=4):
    rng_d = np.random.default_rng(semente)
    erro_knn = np.zeros(len(ks))
    erro_lin = 0.0
    for _ in range(n_rep):
        Xtr = rng_d.uniform(A, B, size=(n_tr, p))
        ytr = r(Xtr[:, 0]) + rng_d.normal(0, SIGMA, size=n_tr)
        Xte = rng_d.uniform(A, B, size=(n_te, p))
        yte_medio = r(Xte[:, 0])           # comparamos com r, e somamos sigma^2

        lin = skl.LinearRegression().fit(Xtr, ytr)
        erro_lin += np.mean((lin.predict(Xte) - yte_medio) ** 2)
        for j, k in enumerate(ks):
            knn = KNeighborsRegressor(n_neighbors=k).fit(Xtr, ytr)
            erro_knn[j] += np.mean((knn.predict(Xte) - yte_medio) ** 2)
    return erro_knn / n_rep + SIGMA ** 2, erro_lin / n_rep + SIGMA ** 2


ks_d = np.array([1, 2, 3, 5, 10, 20, 50, 100, 200])
ps = [1, 2, 4, 10, 20]
saida = {p: experimento_dimensao(p, ks_d) for p in ps}

# a referencia: o preditor constante, que nao aprende nada
xx = np.linspace(A, B, 200_000)
risco_constante = SIGMA ** 2 + r(xx).var()

linha = []
for p in ps:
    knn_p, lin_p = saida[p]
    linha.append({"p": p, "melhor KNN": knn_p.min(), "k otimo": ks_d[np.argmin(knn_p)],
                  "regressao linear": lin_p,
                  "vantagem do KNN": lin_p / knn_p.min()})
print(f"risco de quem chuta a media (nao aprende nada): {risco_constante:.4f}\n")
pd.DataFrame(linha).set_index("p").round(3)

In [ ]:
fig, axes = subplots(1, len(ps), figsize=(11, 2.6), sharey=True)
for ax, p in zip(axes, ps):
    knn_p, lin_p = saida[p]
    ax.plot(1 / ks_d, knn_p, "o-", ms=3.5, color="green", label="KNN")
    ax.axhline(lin_p, ls="--", color="black", lw=1.2, label="regressao linear")
    ax.axhline(risco_constante, ls=":", color="gray", lw=1.2, label="chutar a media")
    ax.set_xscale("log"); ax.set_title(f"p = {p}", fontsize=9)
    ax.set_xlabel("1/k")
axes[0].set_ylabel("risco"); axes[0].legend(fontsize=7)

Com $p=1$ o KNN ganha com folga — a verdade é curva e ele acompanha, cortando
quase pela metade o risco da reta. A vantagem então **mingua monotonicamente**:
1,9× em $p=1$, 1,3× em $p=4$, e a partir de $p=10$ ela vira desvantagem. Em
$p=20$ a regressão linear ganha, apesar de não conseguir **nem representar** a
função verdadeira.

O motivo não é a regressão linear ter melhorado. Ela nem sabe que as variáveis 2
a $p$ são lixo, e paga um pouco de variância por cada uma. O que acontece é que o
KNN piora muito mais rápido. Com 300 pontos espalhados em $[-3,3]^{20}$, os
"$k$ vizinhos mais próximos" de um ponto não são próximos de coisa nenhuma: a
média local deixa de ser local, e o viés explode.

E olhe a linha pontilhada: em $p=20$ os dois métodos já estão a menos de 15% do
risco de quem simplesmente chuta a média, sem olhar covariável nenhuma. Com essa
dimensão e essa amostra, quase não há aprendizado a extrair — e a curva do KNN
fica achatada, sinal de que nenhum $k$ resolve o problema.

Isso tem nome — **maldição da dimensionalidade** — e é a Aula 05 inteira. O que
esta figura antecipa é a moral: *supor uma forma errada pode custar menos que não
supor forma alguma*, e quanto maior $p$, mais isso vale.

---
## 6. No mundo real: KNN no `superconductivity.csv`

Duas advertências práticas para fechar. A primeira é a que a Aula 05 vai
formalizar: aqui $p = 81$, e você já sabe o que isso significa para um método de
vizinhança. A segunda é a escala.

In [ ]:
import os

_nome = "superconductivity.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

df = pd.read_csv(_caminho)
Xs = df.drop(columns="critical_temp").values
ys = df["critical_temp"].values

# o KNN gasta o tempo na PREDICAO, e ela custa O(n_treino) por consulta.
# Uma subamostra de 4000 pontos mantem o notebook rodando em segundos.
rng_s = np.random.default_rng(0)
sub = rng_s.choice(len(ys), size=4000, replace=False)
X_tr, X_te, y_tr, y_te = skm.train_test_split(Xs[sub], ys[sub], test_size=0.3,
                                              random_state=0)
print("treino:", X_tr.shape, "  teste:", X_te.shape)

### A escala não é preciosismo

As 81 colunas estão em unidades completamente diferentes — massa atômica em
unidades de massa atômica, condutividade térmica em W/(m·K), e por aí. A
distância euclidiana soma tudo, então a coluna de maior amplitude decide sozinha
quem é vizinho de quem.

In [ ]:
amplitudes = Xs.max(axis=0) - Xs.min(axis=0)
print(f"amplitude das 81 colunas: de {amplitudes.min():.3g} a {amplitudes.max():.3g}")
print(f"razao entre a maior e a menor: {amplitudes.max() / amplitudes.min():.3g}")

from sklearn.metrics import mean_squared_error

sem_escala = KNeighborsRegressor(n_neighbors=9).fit(X_tr, y_tr)
com_escala = Pipeline([("escala", StandardScaler()),
                       ("knn", KNeighborsRegressor(n_neighbors=9))]).fit(X_tr, y_tr)

print(f"\nEQM no teste, KNN k=9 SEM padronizar: "
      f"{mean_squared_error(y_te, sem_escala.predict(X_te)):.2f}")
print(f"EQM no teste, KNN k=9 COM padronizar: "
      f"{mean_squared_error(y_te, com_escala.predict(X_te)):.2f}")

E a padronização precisa acontecer **dentro** do `Pipeline`, não antes da
validação cruzada: assim ela é refeita em cada dobra, usando só o treino daquela
dobra. É a regra da Aula 03 contra vazamento, e a Aula 07 volta ao assunto.

In [ ]:
modelo = Pipeline([("escala", StandardScaler()), ("knn", KNeighborsRegressor())])
busca = skm.GridSearchCV(modelo, {"knn__n_neighbors": [1, 3, 5, 9, 15, 25, 50]},
                         cv=skm.KFold(5, shuffle=True, random_state=0),
                         scoring="neg_mean_squared_error")
busca.fit(X_tr, y_tr)

# a Ridge da Aula 03, tambem com o hiperparametro escolhido por CV -- comparar
# um modelo ajustado com um modelo no chute nao diria nada
ridge = skm.GridSearchCV(
    Pipeline([("escala", StandardScaler()), ("ridge", skl.Ridge())]),
    {"ridge__alpha": np.logspace(-2, 4, 20)},
    cv=skm.KFold(5, shuffle=True, random_state=0),
    scoring="neg_mean_squared_error").fit(X_tr, y_tr)

print(f"melhor k    : {busca.best_params_['knn__n_neighbors']}")
print(f"melhor alpha: {ridge.best_params_['ridge__alpha']:.4g}")
print(f"\nEQM no teste, KNN   : {mean_squared_error(y_te, busca.predict(X_te)):.2f}")
print(f"EQM no teste, Ridge : {mean_squared_error(y_te, ridge.predict(X_te)):.2f}")
print(f"variancia de y      : {y_te.var():.2f}")

Aqui o KNN ganha, e ganha de longe — em $p = 81$, logo depois de a Seção 5 ter
mostrado o KNN perdendo em $p = 20$. Isso não é contradição, é a informação mais
útil desta seção.

A conta da Seção 5 usava 20 covariáveis **independentes**: cada uma acrescentava
uma direção nova ao espaço, e o volume a preencher multiplicava. As 81 colunas do
`superconductivity.csv` são outra coisa. São todas funções das mesmas
propriedades atômicas dos elementos da fórmula química — média, média ponderada,
média geométrica, entropia, faixa, desvio-padrão de cada propriedade. Elas são
fortemente redundantes, e os materiais não ocupam $[\,\cdot\,]^{81}$: ficam numa
região de dimensão efetiva muito menor.

O que mata o KNN não é o número de colunas, é a **dimensão efetiva** dos dados. A
Aula 05 dá nome a essa distinção e mostra as duas rotas de fuga: quando poucas
covariáveis importam (esparsidade) e quando muitas covariáveis descrevem poucas
direções (redundância).

A Seção 5 mostrou o KNN perdendo com 20 covariáveis independentes e a Seção
anterior o mostrou ganhando com 81 redundantes. Se o que mata é a dimensão efetiva,
ficar com as 5 colunas mais correlacionadas com a resposta deveria andar para a
esquerda naquele painel. Vamos ver.

In [ ]:
cors = np.array([abs(np.corrcoef(X_tr[:, j], y_tr)[0, 1]) for j in range(X_tr.shape[1])])
top5 = np.argsort(cors)[::-1][:5]

print("as 5 mais correlacionadas com critical_temp:")
for j in top5:
    print(f"   {df.columns[j]:32s} |cor| = {cors[j]:.3f}")
print()

for rotulo, cols in [("81 colunas", slice(None)), ("5 colunas", top5)]:
    knn_c = skm.GridSearchCV(
        Pipeline([("escala", StandardScaler()), ("knn", KNeighborsRegressor())]),
        {"knn__n_neighbors": [1, 3, 5, 9, 15, 25, 50]},
        cv=skm.KFold(5, shuffle=True, random_state=0),
        scoring="neg_mean_squared_error").fit(X_tr[:, cols], y_tr)
    ridge_c = skm.GridSearchCV(
        Pipeline([("escala", StandardScaler()), ("ridge", skl.Ridge())]),
        {"ridge__alpha": np.logspace(-2, 4, 20)},
        cv=skm.KFold(5, shuffle=True, random_state=0),
        scoring="neg_mean_squared_error").fit(X_tr[:, cols], y_tr)
    print(f"{rotulo:11s}  KNN (k={knn_c.best_params_['knn__n_neighbors']:2d}) "
          f"EQM {mean_squared_error(y_te, knn_c.predict(X_te[:, cols])):7.2f}"
          f"    Ridge EQM {mean_squared_error(y_te, ridge_c.predict(X_te[:, cols])):7.2f}")

**Os dois pioram, e o KNN piora mais.** O EQM do KNN sai de $179$ para $268$ (50% a
mais); o da Ridge, de $316$ para $487$ (54% a mais).

Não andamos para a esquerda no painel da Seção 5 — andamos para fora dele. Aquele
painel compara métodos com **a mesma informação disponível**, variando só a
dimensão. Aqui jogamos fora 76 colunas, e com elas o sinal que carregavam. As cinco
que sobraram são todas medidas de condutividade térmica e raio atômico; a
correlação individual de $0{,}73$ da melhor delas não diz nada sobre o que as
outras 76 acrescentam **em conjunto**.

O ponto da Seção 5 continua de pé — o que mata o KNN é a dimensão efetiva, não o
número de colunas —, mas ele não implica que reduzir colunas ajude. Reduzir
dimensão só ajuda quando o que sai é ruído. Aqui saiu sinal, e os dois métodos
pagaram.

De passagem, note que o KNN continua batendo a Ridge com folga nos dois recortes.
A relação entre `critical_temp` e essas covariáveis não é linear, e nenhum
subconjunto de colunas conserta isso.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| base truncada | §3 | $(x-t_j)_+^k$ é nula antes do nó — daí a localidade; um dado na ponta mexe pouco no outro lado |
| B-splines | §3 | mesmo espaço da base truncada: ajuste idêntico a $10^{-14}$ |
| KNN | §4 | $k$ **pequeno** é o flexível; a curva é escada por construção |
| custo | §4 | preguiçoso: ajusta em milissegundos, prevê devagar |
| dimensão | §5 | a vantagem do KNN cai de 1,9× ($p=1$) a menos de 1 ($p\ge10$) |
| escala | §6 | sem `StandardScaler`, a coluna de maior amplitude decide quem é vizinho |
| dimensão efetiva | §6 | em $p=81$ **redundantes** o KNN volta a ganhar — o que conta não é o número de colunas |

**Leitura recomendada.** [AME] §4.1–4.2 (séries e *splines*, com a base truncada
que montamos na Seção 3) e §4.3 (KNN e a Figura 4.2 sobre o efeito de $k$).
[ISLP] §3.5 — o experimento da Seção 5 é a Figura 3.20 deles — e o Capítulo 7,
em especial §7.4–7.5.

**Para praticar.** `Lista teorica 04.pdf` (teórica, com gabarito) e
`Lista prática 04.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 05 explica, com teoria, o que a Seção 5 mostrou com uma
simulação: por que a taxa de convergência de todo método de vizinhança degrada
como $n^{-2/(2+p)}$, e o que ainda dá para fazer quando $p$ é grande.